# 🔧 Módulo 4 (visual) — Fine-tuning LoRA com comparação antes/depois

Acompanha o **Módulo 4** do guia (`docs/02_GUIA_DE_APRENDIZADO.md`).
Aqui você vai:
1. Perguntar algo ao modelo **base** (antes do treino);
2. Rodar o **fine-tuning LoRA** capturando a curva de loss ao vivo;
3. Plotar a curva de treino/validação;
4. Perguntar de novo ao modelo **ajustado** e comparar as respostas.

> Abra com `make lab` a partir da raiz, com o `.venv` ativo. Requer os dados do
> Módulo 1 (`make data`). Na 1ª vez baixa o modelo base (~300 MB).

## 1. Setup e configuração

In [ ]:
import os
import re
import sys
import subprocess

import matplotlib.pyplot as plt

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
os.chdir(ROOT)  # rodamos os comandos a partir da raiz do projeto

MODEL = "mlx-community/Qwen2.5-0.5B-Instruct-4bit"
ADAPTER_DIR = "models/finetuned/lora-adapters-nb"
ITERS = 100          # aumente para treinar mais
PERGUNTA = "O que é LoRA? Responda em uma frase."

assert os.path.exists("data/processed/train.jsonl"), "Rode 'make data' antes!"
print("Projeto:", ROOT)
print("Modelo base:", MODEL)

## 2. Resposta do modelo BASE (antes do fine-tuning)

Definimos um ajudante que chama `mlx_lm.generate` e captura o texto gerado.

In [ ]:
def responder(prompt, adapter=None, max_tokens=80):
    cmd = [sys.executable, "-m", "mlx_lm", "generate",
           "--model", MODEL, "--prompt", prompt,
           "--max-tokens", str(max_tokens), "--temp", "0.3"]
    if adapter:
        cmd += ["--adapter-path", adapter]
    out = subprocess.run(cmd, capture_output=True, text=True).stdout
    # o texto gerado fica entre as linhas de '==========' do mlx_lm
    parts = out.split("==========")
    return parts[1].strip() if len(parts) >= 2 else out.strip()

resposta_antes = responder(PERGUNTA)
print("ANTES (modelo base):\n")
print(resposta_antes)

## 3. Fine-tuning LoRA capturando a loss ao vivo

Rodamos `mlx_lm lora` como subprocesso e usamos regex para extrair as linhas
`Iter N: Train loss ...` e `Iter N: Val loss ...`, guardando em listas para
plotar. É o mesmo treino do `scripts/03_finetune_lora.sh`, só que instrumentado.

In [ ]:
cmd = [sys.executable, "-m", "mlx_lm", "lora",
       "--model", MODEL, "--train",
       "--data", "data/processed",
       "--adapter-path", ADAPTER_DIR,
       "--iters", str(ITERS),
       "--batch-size", "4",
       "--num-layers", "8",
       "--learning-rate", "1e-4",
       "--steps-per-report", "10",
       "--steps-per-eval", "20"]

re_train = re.compile(r"Iter (\d+): Train loss ([\d.]+)")
re_val   = re.compile(r"Iter (\d+): Val loss ([\d.]+)")

train_x, train_y, val_x, val_y = [], [], [], []

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    mt = re_train.search(line)
    mv = re_val.search(line)
    if mt:
        train_x.append(int(mt.group(1))); train_y.append(float(mt.group(2)))
        print(f"  treino  iter {mt.group(1):>4} | loss {mt.group(2)}")
    if mv:
        val_x.append(int(mv.group(1))); val_y.append(float(mv.group(2)))
        print(f"  val     iter {mv.group(1):>4} | loss {mv.group(2)}")
proc.wait()
print("\nFine-tuning concluído. Adaptador salvo em", ADAPTER_DIR)

## 4. 📈 Curva de loss do fine-tuning

In [ ]:
plt.figure(figsize=(8, 4))
if train_x:
    plt.plot(train_x, train_y, label="train", marker="o", ms=4)
if val_x:
    plt.plot(val_x, val_y, label="val", marker="s", ms=4)
plt.title(f"LoRA fine-tuning — {MODEL.split('/')[-1]}")
plt.xlabel("iteração")
plt.ylabel("loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Resposta DEPOIS (com o adaptador) e comparação

Note que só ~0.3% dos pesos foram treinados (o adaptador LoRA), mas o estilo da
resposta já reflete os dados de treino.

In [ ]:
resposta_depois = responder(PERGUNTA, adapter=ADAPTER_DIR)

print("=" * 70)
print("PERGUNTA:", PERGUNTA)
print("=" * 70)
print("\n🔹 ANTES (modelo base):\n")
print(resposta_antes)
print("\n🔸 DEPOIS (com LoRA):\n")
print(resposta_depois)
print("=" * 70)

## 6. 🧪 Exercícios

1. Aumente `ITERS` para 300 e re-execute as células 3→5. A loss cai mais? A
   resposta muda?
2. Edite `data/processed/train.jsonl` com **seus** pares de Q&A, rode `make data`
   com seus dados e re-treine.
3. Troque `MODEL` por `mlx-community/Qwen2.5-1.5B-Instruct-4bit` e compare.
4. **Fundir** o adaptador num modelo standalone:
   ```bash
   python -m mlx_lm fuse --model mlx-community/Qwen2.5-0.5B-Instruct-4bit \
       --adapter-path models/finetuned/lora-adapters-nb \
       --save-path models/finetuned/merged-nb
   ```

📝 Registre no `docs/03_DIARIO.md`.